In [1]:
import os
import pandas as pd
import tensorflow as tf
import cv2
import random
from sklearn.model_selection import train_test_split

data_dir = './'  # 영상 데이터 폴더 경로
label_dirs = ['raw', 'deepfake', 'filter']  # 라벨 폴더 이름

video_paths = []
labels = []

for label_dir in label_dirs:
    label_path = os.path.join(data_dir, label_dir)
    for video_file in os.listdir(label_path):
        if video_file.endswith('.mp4'):
            video_paths.append(os.path.join(label_path, video_file))
            labels.append(label_dir)

df = pd.DataFrame({'video_path': video_paths, 'label': labels})

In [2]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

In [3]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # GPU 메모리 증가 허용
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 11019011454628125932
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 7783579648
locality {
  bus_id: 1
  links {
  }
}
incarnation: 3772276007405653582
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]
1 Physical GPUs, 1 Logical GPUs


In [4]:
def load_video(video_path, target_size=(224, 224), num_frames=30):
    """동영상에서 랜덤 프레임을 추출하고 전처리합니다."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return tf.zeros((num_frames, target_size[0], target_size[1], 3), dtype=tf.float32)

    # 랜덤 프레임 인덱스 생성
    random_indices = sorted(random.sample(range(frame_count), min(num_frames, frame_count)))

    for index in random_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, index)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, target_size)
        frame = frame / 255.0
        frames.append(frame)
    cap.release()

    # 프레임 수가 부족하면 패딩 추가
    while len(frames) < num_frames:
        frames.append(tf.zeros((target_size[0], target_size[1], 3), dtype=tf.float32))

    return tf.convert_to_tensor(frames, dtype=tf.float32)

def encode_label(label):
    """라벨을 one-hot encoding으로 변환합니다."""
    labels = ['raw', 'deepfake', 'filter']
    one_hot = [1.0 if l == label else 0.0 for l in labels]
    return tf.convert_to_tensor(one_hot, dtype=tf.float32)

def create_dataset(df, batch_size=8):
    """데이터셋을 생성합니다."""
    video_paths = df['video_path'].tolist()
    labels = df['label'].tolist()

    def generator():
        for path, label in zip(video_paths, labels):
            video = load_video(path)
            encoded_label = encode_label(label)
            yield video, encoded_label

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(tf.TensorSpec(shape=(30, 224, 224, 3), dtype=tf.float32),  # 고정된 프레임 수
                          tf.TensorSpec(shape=(3,), dtype=tf.float32)))
    dataset = dataset.batch(batch_size)
    return dataset

# 학습 데이터셋 생성
train_dataset = create_dataset(train_df).prefetch(tf.data.AUTOTUNE)
# 테스트 데이터셋 생성
test_dataset = create_dataset(test_df).prefetch(tf.data.AUTOTUNE)

In [5]:
from tensorflow.keras.layers import Conv2D, GlobalAveragePooling1D, AveragePooling2D, Flatten, Dense, Input
from tensorflow.keras.models import Model

def Meso4(input_shape=(224, 224, 3)):
    input = Input(shape=input_shape)

    # Meso4 모델 구조 정의
    x = Conv2D(8, 3, padding='same', activation='relu')(input)
    x = Conv2D(8, 5, padding='same', activation='relu')(x)
    x = AveragePooling2D(pool_size=(2, 2))(x)

    x = Conv2D(16, 3, padding='same', activation='relu')(x)
    x = Conv2D(16, 5, padding='same', activation='relu')(x)
    x = AveragePooling2D(pool_size=(2, 2))(x)

    x = Conv2D(16, 5, padding='same', activation='relu')(x)
    x = Conv2D(16, 5, padding='same', activation='relu')(x)
    x = AveragePooling2D(pool_size=(2, 2))(x)

    x = Conv2D(16, 5, padding='same', activation='relu')(x)
    x = Conv2D(16, 5, padding='same', activation='relu')(x)
    x = AveragePooling2D(pool_size=(2, 2))(x)

    x = Flatten()(x)
    x = Dense(16, activation='relu')(x)
    output = Dense(3, activation='softmax')(x)

    return Model(inputs=input, outputs=output)

# MesoNet 모델 기반 동영상 분류 모델 정의
with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    input_video = Input(shape=(30, 224, 224, 3))
    frames = tf.unstack(input_video, axis=1)  # 프레임을 시간 축에서 분리

    frame_outputs = []
    for frame in frames:
        frame_output = Meso4()(frame)
        frame_outputs.append(frame_output)

    x = tf.stack(frame_outputs, axis=1)  # 분리된 프레임 결과를 다시 시간 축으로 합침
    x = GlobalAveragePooling1D()(x) # 시간축 평균을 냄 (1D 적용)
    output = Dense(3, activation='softmax')(x)

    model = Model(inputs=input_video, outputs=output)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
# 모델 학습
model.fit(train_dataset, epochs=5, validation_data=test_dataset)
model.save('Mesonet_classification_model.keras')


Epoch 1/5
143/143 [==============================] - 4524s 32s/step - loss: 1.1189 - accuracy: 0.3877 - val_loss: 1.0826 - val_accuracy: 0.3965
Epoch 2/5
143/143 [==============================] - 4000s 28s/step - loss: 1.0892 - accuracy: 0.3421 - val_loss: 1.0912 - val_accuracy: 0.3368
Epoch 3/5
143/143 [==============================] - 3974s 28s/step - loss: 1.0837 - accuracy: 0.3939 - val_loss: 1.0806 - val_accuracy: 0.5228
Epoch 4/5
143/143 [==============================] - 3971s 28s/step - loss: 1.0873 - accuracy: 0.3939 - val_loss: 1.0971 - val_accuracy: 0.3544
Epoch 5/5
143/143 [==============================] - 8351s 59s/step - loss: 1.0968 - accuracy: 0.3351 - val_loss: 1.0918 - val_accuracy: 0.3544


In [6]:
# 모델 평가
loss, accuracy = model.evaluate(test_dataset)
print(f"Test Accuracy: {accuracy}")

36/36 [==============================] - 2286s 63s/step - loss: 1.0919 - accuracy: 0.3404
Test Accuracy: 0.340350866317749
